In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet(
    "../data/test_datasets/mbd_dataset/detail/trx/fold=0"
)

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7634228 entries, 0 to 7634227
Data columns (total 14 columns):
 #   Column         Dtype         
---  ------         -----         
 0   client_id      object        
 1   event_time     datetime64[ns]
 2   amount         float32       
 3   event_type     int32         
 4   event_subtype  int32         
 5   currency       float64       
 6   src_type11     float64       
 7   src_type12     float64       
 8   dst_type11     float64       
 9   dst_type12     float64       
 10  src_type21     float64       
 11  src_type22     float64       
 12  src_type31     float64       
 13  src_type32     float64       
dtypes: datetime64[ns](1), float32(1), float64(9), int32(2), object(1)
memory usage: 728.1+ MB


In [3]:
df["year_month"] = df["event_time"].dt.to_period("M")
# для учёта сезонности потом
df["month"] = df["year_month"].dt.month

In [4]:
client_month_features = df.groupby(["client_id", "year_month"]).agg(
    # Для валюты 11 
    amount_11_sum=("amount", lambda x: x[df.loc[x.index, "currency"] == 11].sum()),
    amount_11_mean=("amount", lambda x: x[df.loc[x.index, "currency"] == 11].mean()),
    amount_11_median=("amount", lambda x: x[df.loc[x.index, "currency"] == 11].median()),
    amount_11_std=("amount", lambda x: x[df.loc[x.index, "currency"] == 11].std()),
    amount_11_min=("amount", lambda x: x[df.loc[x.index, "currency"] == 11].min()),
    amount_11_max=("amount", lambda x: x[df.loc[x.index, "currency"] == 11].max()),
    tx_11_count=("currency", lambda x: (x == 11).sum()),
    
    # общее количество транзакций
    tx_total=("amount", "count")
).reset_index()

In [5]:
client_month_features = client_month_features.sort_values(["client_id", "year_month"])
for lag in [1, 2, 3, 6, 9, 12]:
    client_month_features[f"lag{lag}_amount_11_sum"] = client_month_features.groupby("client_id")["amount_11_sum"].shift(lag)
    client_month_features[f"lag{lag}_amount_11_mean"] = client_month_features.groupby("client_id")["amount_11_mean"].shift(lag)
    client_month_features[f"lag{lag}_amount_11_median"] = client_month_features.groupby("client_id")["amount_11_median"].shift(lag)
    client_month_features[f"lag{lag}_amount_11_std"] = client_month_features.groupby("client_id")["amount_11_std"].shift(lag)
    client_month_features[f"lag{lag}_amount_11_min"] = client_month_features.groupby("client_id")["amount_11_min"].shift(lag)
    client_month_features[f"lag{lag}_tx_11_count"] = client_month_features.groupby("client_id")["tx_11_count"].shift(lag)
    client_month_features[f"lag{lag}_tx_total"] = client_month_features.groupby("client_id")["tx_total"].shift(lag)

for window in [3, 6, 12]:
    # Сумма
    client_month_features[f"amount_11_sum_rolling_mean_{window}m"] = (
        client_month_features.groupby("client_id")["amount_11_sum"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    
    # Количество транзакций
    client_month_features[f"tx_11_count_rolling_mean_{window}m"] = (
        client_month_features.groupby("client_id")["tx_11_count"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    
    # Общее количество транзакций
    client_month_features[f"tx_total_rolling_mean_{window}m"] = (
        client_month_features.groupby("client_id")["tx_total"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    
    # Максимум за период
    client_month_features[f"amount_11_sum_rolling_max_{window}m"] = (
        client_month_features.groupby("client_id")["amount_11_sum"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).max())
    )
    
    # Минимум за период
    client_month_features[f"amount_11_sum_rolling_min_{window}m"] = (
        client_month_features.groupby("client_id")["amount_11_sum"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).min())
    )
    
    # Стандартное отклонение (волатильность)
    client_month_features[f"amount_11_sum_rolling_std_{window}m"] = (
        client_month_features.groupby("client_id")["amount_11_sum"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).std())
    )

In [6]:
# Накопленная сумма (вся история до текущего месяца)
client_month_features["amount_11_cumsum"] = (
    client_month_features
        .groupby("client_id")["amount_11_sum"]
        .apply(lambda x: x.cumsum().shift(1))
        .reset_index(level=0, drop=True)
)

# Накопленное количество транзакций в валюте 11
client_month_features["tx_11_cumsum"] = (
    client_month_features
        .groupby("client_id")["tx_11_count"]
        .apply(lambda x: x.cumsum().shift(1))
        .reset_index(level=0, drop=True)
)

# Накопленное общее количество транзакций
client_month_features["tx_total_cumsum"] = (
    client_month_features
        .groupby("client_id")["tx_total"]
        .apply(lambda x: x.cumsum().shift(1))
        .reset_index(level=0, drop=True)
)


# Средний чек накопленным итогом (до текущего месяца)
client_month_features["amount_11_cummean"] = (
    client_month_features["amount_11_cumsum"] / 
    client_month_features["tx_11_cumsum"].replace(0, np.nan)
)

In [7]:
# до текущего месяца

# Максимум за всю историю (до текущего месяца)
client_month_features["amount_11_hist_max"] = (
    client_month_features.groupby("client_id")["amount_11_sum"]
    .transform(lambda x: x.shift(1).expanding().max())
)

# Минимум за всю историю
client_month_features["amount_11_hist_min"] = (
    client_month_features.groupby("client_id")["amount_11_sum"]
    .transform(lambda x: x.shift(1).expanding().min())
)

# Среднее за всю историю
client_month_features["amount_11_hist_mean"] = (
    client_month_features.groupby("client_id")["amount_11_sum"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

# Стандартное отклонение за всю историю
client_month_features["amount_11_hist_std"] = (
    client_month_features.groupby("client_id")["amount_11_sum"]
    .transform(lambda x: x.shift(1).expanding().std())
)
# Доля транзакций в валюте 11 накопленным итогом
client_month_features["tx_11_share_cum"] = (
    client_month_features["tx_11_cumsum"] / 
    client_month_features["tx_total_cumsum"]
)

In [8]:
# Извлекаем месяц и квартал
client_month_features["month"] = client_month_features["year_month"].dt.month
client_month_features["quarter"] = client_month_features["year_month"].dt.quarter

# Среднее за этот же месяц в прошлом году (если данных достаточно)
client_month_features["amount_11_same_month_last_year"] = (
    client_month_features
        .groupby("client_id")["amount_11_sum"]
        .shift(12)
)

# Флаг начала квартала
client_month_features["is_quarter_start"] = (client_month_features["month"] % 3 == 1).astype(int)

# Флаг конца квартала
client_month_features["is_quarter_end"] = (client_month_features["month"] % 3 == 0).astype(int)


In [9]:
# Безопасные динамические признаки через лаги

# Используем lag1_amount_11_sum и lag2_amount_11_sum
# lag1 — сумма за прошлый месяц
# lag2 — сумма за месяц до прошлого (т.е. два месяца назад)

# Изменение относительно прошлого месяца (процентное)
client_month_features["amount_11_pct_change_lag"] = (
    (client_month_features["lag1_amount_11_sum"] - client_month_features["lag2_amount_11_sum"])
    / client_month_features["lag2_amount_11_sum"].replace(0, np.nan) * 100
)

# Изменение относительно прошлого месяца (абсолютное)
client_month_features["amount_11_abs_change_lag"] = (
    client_month_features["lag1_amount_11_sum"] - client_month_features["lag2_amount_11_sum"]
)

# Отношение к среднему за последние 3 месяца (rolling3 уже сдвинуто на 1 месяц)
client_month_features["amount_11_vs_3m_avg_lag"] = (
    client_month_features["lag1_amount_11_sum"] / client_month_features["amount_11_sum_rolling_mean_3m"].replace(0, np.nan)
)

# Отношение к среднему за всю историю (cumulative)
client_month_features["amount_11_vs_cummean_lag"] = (
    client_month_features["lag1_amount_11_sum"] / client_month_features["amount_11_cummean"].replace(0, np.nan)
)

# Отношение к максимуму за всю историю
client_month_features["amount_11_vs_hist_max_lag"] = (
    client_month_features["lag1_amount_11_sum"] / client_month_features["amount_11_hist_max"].replace(0, np.nan)
)

# Отношение к минимуму за всю историю
client_month_features["amount_11_vs_hist_min_lag"] = (
    client_month_features["lag1_amount_11_sum"] / client_month_features["amount_11_hist_min"].replace(0, np.nan)
)

# Коэффициент вариации (CV = std/mean) — уже безопасен, так как hist_std и hist_mean строятся через shift(1)
client_month_features["amount_11_hist_cv_lag"] = (
    client_month_features["amount_11_hist_std"] / client_month_features["amount_11_hist_mean"].replace(0, np.nan)
)

# Размах (max - min)
client_month_features["amount_11_hist_range_lag"] = (
    client_month_features["amount_11_hist_max"] - client_month_features["amount_11_hist_min"]
)

# Был ли всплеск (превышение среднего на 2 сигмы)
client_month_features["amount_11_is_spike_lag"] = (
    (client_month_features["lag1_amount_11_sum"] > 
     client_month_features["amount_11_hist_mean"] + 2 * client_month_features["amount_11_hist_std"])
).astype(int)

# Был ли провал
client_month_features["amount_11_is_dip_lag"] = (
    (client_month_features["lag1_amount_11_sum"] < 
     client_month_features["amount_11_hist_mean"] - 2 * client_month_features["amount_11_hist_std"])
).astype(int)


# Количество месяцев подряд с положительной динамикой
def count_consecutive_positives_lag(x):
    """Считает, сколько месяцев подряд сумма росла,
       используем lag1 для текущей позиции, lag2 для сравнения"""
    result = []
    count = 0
    for i in range(len(x)):
        if i > 1 and x.iloc[i-1] > x.iloc[i-2]:  # сравниваем lag1 и lag2
            count += 1
        else:
            count = 0
        result.append(count)
    return pd.Series(result, index=x.index)

client_month_features["amount_11_consecutive_growth_lag"] = (
    client_month_features.groupby("client_id")["lag1_amount_11_sum"]
    .transform(lambda x: count_consecutive_positives_lag(x))
)


# Нулевые транзакции и доля активности
# Был ли месяц с нулевыми транзакциями (через lag1)
client_month_features["amount_11_is_zero_lag"] = (client_month_features["lag1_amount_11_sum"] == 0).astype(int)

# Доля месяцев с активностью (накопленным итогом), безопасно через shift(1)
client_month_features["amount_11_active_ratio_lag"] = (
    client_month_features.groupby("client_id")["amount_11_is_zero_lag"]
    .transform(lambda x: 1 - x.shift(1).expanding().mean())
)

In [10]:
client_month_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 364817 entries, 0 to 364816
Data columns (total 97 columns):
 #   Column                            Non-Null Count   Dtype    
---  ------                            --------------   -----    
 0   client_id                         364817 non-null  object   
 1   year_month                        364817 non-null  period[M]
 2   amount_11_sum                     364817 non-null  float32  
 3   amount_11_mean                    364815 non-null  float64  
 4   amount_11_median                  364815 non-null  float64  
 5   amount_11_std                     329754 non-null  float64  
 6   amount_11_min                     364815 non-null  float64  
 7   amount_11_max                     364815 non-null  float64  
 8   tx_11_count                       364817 non-null  int64    
 9   tx_total                          364817 non-null  int64    
 10  lag1_amount_11_sum                344785 non-null  float32  
 11  lag1_amount_11_mean       

In [11]:
# Event type признаки (кумулятивно до текущего месяца)
# Считаем количество транзакций каждого типа по клиенту и месяцу
event_counts_monthly = (
    df.groupby(["client_id", "year_month", "event_type"])["amount"]
    .count()
    .unstack(fill_value=0)
)

# Чтобы получить кумулятивные признаки до месяца M, сдвигаем на 1 месяц
event_counts_cumsum = event_counts_monthly.groupby(level=0).cumsum().shift(fill_value=0)

# Считаем долю каждого типа от всех транзакций до текущего месяца
event_shares_cumsum = event_counts_cumsum.div(event_counts_cumsum.sum(axis=1), axis=0)

# Берем топ-10 типов событий, остальные объединяем в "other_events"
top_events_cumsum = event_shares_cumsum.iloc[:, :10].copy()
top_events_cumsum["other_events"] = 1 - top_events_cumsum.sum(axis=1)

# Сбрасываем индекс для объединения с другими признаками
event_features = top_events_cumsum.reset_index()


# src/dst признаки (кумулятивно до текущего месяца)
src_cols = ["src_type11", "src_type12", "src_type21", "src_type22", "src_type31", "src_type32"]
dst_cols = ["dst_type11", "dst_type12"]

# Базовая таблица с client_id и год-месяцем
src_dst_features = df[["client_id", "year_month"]].drop_duplicates().copy()

for col in src_cols + dst_cols:
    # Считаем уникальные значения по клиенту и месяцу
    stats = (
        df.groupby(["client_id", "year_month"])[col]
        .nunique()
        .groupby(level=0)       # группируем по клиенту
        .cumsum()               # кумулятивная сумма по месяцам
        .shift(fill_value=0)    # сдвигаем на 1 месяц, чтобы исключить текущий
        .rename(f"{col}_n_unique")
        .reset_index()
    )
    # Объединяем с базовой таблицей
    src_dst_features = src_dst_features.merge(stats, on=["client_id", "year_month"], how="left")


In [12]:
src_dst_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 364817 entries, 0 to 364816
Data columns (total 10 columns):
 #   Column               Non-Null Count   Dtype    
---  ------               --------------   -----    
 0   client_id            364817 non-null  object   
 1   year_month           364817 non-null  period[M]
 2   src_type11_n_unique  364817 non-null  int64    
 3   src_type12_n_unique  364817 non-null  int64    
 4   src_type21_n_unique  364817 non-null  int64    
 5   src_type22_n_unique  364817 non-null  int64    
 6   src_type31_n_unique  364817 non-null  int64    
 7   src_type32_n_unique  364817 non-null  int64    
 8   dst_type11_n_unique  364817 non-null  int64    
 9   dst_type12_n_unique  364817 non-null  int64    
dtypes: int64(8), object(1), period[M](1)
memory usage: 27.8+ MB


In [13]:
client_month_features = client_month_features.merge(src_dst_features, on=["client_id", "year_month"], how="left")

In [14]:
client_month_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 364817 entries, 0 to 364816
Columns: 105 entries, client_id to dst_type12_n_unique
dtypes: float32(11), float64(74), int64(18), object(1), period[M](1)
memory usage: 276.9+ MB


In [15]:
# Сохраняем финальный датасет
version = 2
client_month_features.to_parquet(f"../data/processed/client_month_features_{version}.parquet", index=False)

print(f"Финальный датасет: {client_month_features.shape[0]} строк, {client_month_features.shape[1]} признаков")


Финальный датасет: 364817 строк, 105 признаков


In [ ]:
# не используем для обучения
current_month_features = [
    'amount_11_sum',      # то, что предсказываем
    'amount_11_mean',     # статистика этого месяца
    'amount_11_median',
    'amount_11_std',
    'amount_11_min',
    'amount_11_max',
    'tx_11_count',
    'tx_total'] 